# Full-scale Vecchia approximation parameters

## Packages

In [3]:
import os
import torch
import gpytorch
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.stats import norm
import numpy as np
import json
import gpboost as gpb
import requests
import pandas as pd
import time
from properscoring import crps_gaussian as crps_norm

/usr/local/cuda-13/targets/x86_64-linux/lib


/usr/sepp1.5.1/scratch/tmp/365/FP_data/GPU/space_time_GPU/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [33]:
data = pd.read_parquet("https://raw.githubusercontent.com/TimGyger/SpaceTimeGPApprox/refs/heads/main/Data/Real_World_TEMP_PRCP.parquet", 
                       engine="pyarrow")
data["date"] = pd.to_datetime(data["date"])

# 2. Filter for complete coverage, distinct date/X/Y, and stations with enough dates
data_complete = (
    data
    .drop_duplicates(subset=["date", "X", "Y"])
    .loc[lambda df: df["date"] < "2025-08-01"]
)

# Arrange by year, doy, then group by year & doy to assign t = cur_group_id()
data_complete = data_complete.sort_values(["year", "doy"])
# Create a unique group id per (year, doy)
data_complete["t"] = pd.factorize(list(zip(data_complete["year"], data_complete["doy"])))[0] + 1

# Drop unused columns
#data_complete = data_complete.drop(columns=["year", "count"])
data_complete = data_complete.drop(columns=["year"])
# 3. Build fixed effects / derived features
df = data_complete.copy()
df["prcp"] = df["prcp"]/100
#df.loc[df["prcp"] < 1, "prcp"] = 0
df["KGZones"] = df["KGZones"].astype("category")
df["sin_t"] = np.sin(2 * np.pi * df["doy"] / 366)
df["cos_t"] = np.cos(2 * np.pi * df["doy"] / 366)
df["sin_t2"] = np.sin(4 * np.pi * df["doy"] / 366)
df["cos_t2"] = np.cos(4 * np.pi * df["doy"] / 366)
zone_dummies = pd.get_dummies(df["KGZones"], prefix="KGZones", drop_first=True).astype(float)
for col in zone_dummies.columns:
    df[f"{col}_sin_t"] = zone_dummies[col] * df["sin_t"]
    df[f"{col}_cos_t"] = zone_dummies[col] * df["cos_t"]
    df[f"{col}_sin_t2"] = zone_dummies[col] * df["sin_t2"]
    df[f"{col}_cos_t2"] = zone_dummies[col] * df["cos_t2"]
df = pd.concat([df, zone_dummies], axis=1)
df["X2"] = df["X"]**2
df["Y2"] = df["Y"]**2
df["XY"] = df["X"] * df["Y"]
df["elevation"] = pd.to_numeric(df["elevation"], errors="coerce")
df["elevation2"] = df["elevation"]**2
df["northness"] = np.cos(np.deg2rad(df["aspect_deg"]))
df["eastness"] = np.sin(np.deg2rad(df["aspect_deg"]))
df["slope_north"] = df["slope_deg"] * df["northness"]
df["slope_east"] = df["slope_deg"] * df["eastness"]
df["elevation_slope"] = df["elevation"] * df["slope_deg"]
df["sqrt_distance_to_sea"] = np.sqrt(df["distance_to_sea"])
df["sqrt_distance_to_sea_elevation"] = df["sqrt_distance_to_sea"] * df["elevation"]
df["sin_t_elev"] = df["sin_t"] * df["elevation"]
df["cos_t_elev"] = df["cos_t"] * df["elevation"]
df["sin_t_dist_sea"] = df["sqrt_distance_to_sea"] * df["sin_t"]
df["cos_t_dist_sea"] = df["sqrt_distance_to_sea"] * df["cos_t"]
df["sin_t_X"] = df["X"] * df["sin_t"]
df["cos_t_X"] = df["X"] * df["cos_t"]
df["sin_t_Y"] = df["Y"] * df["sin_t"]
df["cos_t_Y"] = df["Y"] * df["cos_t"]
#df["normals"] = df["DLY-TMAX-NORMAL"]

# Drop columns aspect_deg, DLY-TMAX-NORMAL, doy, distance_to_sea
df = df.drop(columns=["aspect_deg", "DLY-TMAX-NORMAL", "doy", "distance_to_sea", "KGZones","tmax","prcp_binary","id"])

df = df.dropna()
# 4. Split train/test by date, and drop 25% stations randomly from train
np.random.seed(42)

# Extract unique stations (X, Y) prior to 2025‑01‑01
stations = (
    df[df["date"] < "2025-01-01"]
    .drop_duplicates(subset=["X", "Y"])
    .reset_index(drop=True)
)

# Sample 25% of these to remove
stations_to_remove = stations.sample(frac=0.25, random_state=42)



/tmp/ipykernel_131581/1257185525.py:19: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  data_complete["t"] = pd.factorize(list(zip(data_complete["year"], data_complete["doy"])))[0] + 1


In [5]:
# Generate last day of each month for 2025
last_days = pd.date_range("2025-01-01", "2025-12-31", freq="M")

# Compute DOY
doys = last_days.dayofyear.tolist()

# Add 366 to shift to "year 2"
doys_shifted = [d + 366 for d in doys]
doys_shifted = [366] + doys_shifted
doys_shifted

/tmp/ipykernel_131581/1162718386.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  last_days = pd.date_range("2025-01-01", "2025-12-31", freq="M")


[366, 397, 425, 456, 486, 517, 547, 578, 609, 639, 670, 700, 731]

In [6]:
df.head()

,elevation,date,slope_deg,X,Y,prcp,t,sin_t,cos_t,sin_t2,...,sqrt_distance_to_sea,sqrt_distance_to_sea_elevation,sin_t_elev,cos_t_elev,sin_t_dist_sea,cos_t_dist_sea,sin_t_X,cos_t_X,sin_t_Y,cos_t_Y
1023,231.6,2024-01-01,0.142841,780296.044029,1.358085e+06,0.0,1,0.017166,0.999853,0.034328,...,755.109411,174883.339581,3.975722,231.565873,12.962457,754.998144,13394.819198,780181.065616,23313.330993,1.357885e+06
1624,210.0,2024-01-01,0.159490,820260.738802,1.340746e+06,0.0,1,0.017166,0.999853,0.034328,...,743.665070,156169.664726,3.604929,209.969056,12.766000,743.555489,14080.866327,820139.871500,23015.683857,1.340548e+06
2221,91.4,2024-01-01,0.056472,812212.477027,9.373078e+05,0.0,1,0.017166,0.999853,0.034328,...,297.637286,27204.047976,1.569003,91.386532,5.109340,297.593429,13942.707212,812092.795655,16090.134314,9.371697e+05
2795,166.1,2024-01-01,0.381445,981291.864859,1.113023e+06,0.0,1,0.017166,0.999853,0.034328,...,537.518392,89281.804993,2.851327,166.075525,9.227218,537.439188,16845.179738,981147.269251,19106.521549,1.112859e+06
3964,82.6,2024-01-01,0.311809,782293.531797,9.013886e+05,0.0,1,0.017166,0.999853,0.034328,...,187.555758,15492.105623,1.417939,82.587829,3.219644,187.528121,13429.108732,782178.259050,15473.534790,9.012558e+05


In [7]:
def p_nonzero(mu_vec,sigma):
    return norm.cdf(mu_vec / sigma)

def compute_crps_vectorized(y_vec, mu_vec, sigma, lam, m):
    rng = np.random.default_rng(1)  # Fixed RNG seed for reproducibility
    
    # Generate all random samples at once, shape: (len(y_vec), m)
    X = mu_vec[:, None] + sigma * rng.standard_normal((len(y_vec), m))  # Broadcasting
    Y = np.maximum(X, 0.0)  # Apply max(X, 0) for each element
    Y = np.power(Y, lam)    # Apply the power transformation
    
    # Term1: Mean absolute difference between Y and y (broadcasted)
    term1 = np.mean(np.abs(Y - y_vec[:, None]), axis=1)  # Shape: (len(y_vec),)
    
    # Term2: Mean absolute differences between each pair of Y values
    term2 = np.mean(np.abs(Y[:, :, None] - Y[:, None, :]), axis=(1, 2))  # Shape: (len(y_vec),)
    
    # Return CRPS values for each (y, mu) pair
    return term1 - 0.5 * term2

## Experiments

### Vecchia (Euclidean)

In [7]:
cov_pars = [] 
aux_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
pred_latent = [] 
pred_var_latent = [] 
Time_vec = []
cov_par = pd.DataFrame([[ 1, 0.037, 6.4939e-06, 0.5, 0.5, 0.5, 0.5]],
                  columns=["sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[0.5] + [0]*(df.shape[1]-8)])
aux_par = pd.DataFrame([[2.6,3]])
for i in range(366, 572):
    print(i)
    df_train = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] >= "2024-06-30")]
    
    mask = (
        (df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i-100]))
    )

    # Apply filter
    df_filtered = df[mask]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["prcp"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "prcp"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    const_mask = np.all(X_train == X_train[0, :], axis=0)
    const_mask[0] = False
    X_train = X_train[:, ~const_mask]
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 5])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["prcp"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "prcp"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    X_test = X_test[:, ~const_mask]
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "iterative", gp_approx="vecchia")
        model.fit(X = X_train,y = y_train,
                 params={"maxit": 1000,"trace": True, "estimate_cov_par_index": [1,1,1,1,0,1,1], #"estimate_aux_pars": False, 
                         "init_aux_pars": aux_par.to_numpy().ravel().tolist(),
                         "std_dev": False, #"num_rand_vec_trace": 1000,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        print(aux_par)
        print(cov_par)
        cov_pars.append(cov_par)
        aux_pars.append(aux_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "iterative", gp_approx="vecchia")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,0,1,1], 
                          "init_aux_pars": aux_par.to_numpy().ravel().tolist(),"std_dev": False,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])
    pred_Linear_latent_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_response = False,predict_var=True)
    pred_latent.append(pred_Linear_latent_Model["mu"])
    pred_var_latent.append(pred_Linear_latent_Model["var"])

366
2024-09-23 00:00:00
Re-train model
[GPBoost] [Info] Use GPU
Launching kernel with 256 blocks, 693 threads (n=177227)
[GPBoost] [Warning] The linear regression covariate data matrix (fixed effect) is rank deficient. This is not necessarily a problem when using gradient descent. If this is not desired, consider dropping some columns / covariates 
[GPBoost] [Debug] GPModel: initial parameters: 
[GPBoost] [Debug] cov_pars[0]: 1
[GPBoost] [Debug] cov_pars[1]: 0.037
[GPBoost] [Debug] cov_pars[2]: 6.4939e-06
[GPBoost] [Debug] cov_pars[3]: 0.5
[GPBoost] [Debug] cov_pars[4]: 0.5
[GPBoost] [Debug] cov_pars[5]: 0.5
[GPBoost] [Debug] cov_pars[6]: 0.5
[GPBoost] [Debug] beta[0]: 0.5
[GPBoost] [Debug] beta[1]: 0
[GPBoost] [Debug] beta[2]: 0
[GPBoost] [Debug] beta[3]: 0
[GPBoost] [Debug] beta[4]: 0
[GPBoost] [Debug] Note: only the first 5 linear regression coefficients are shown 
[GPBoost] [Debug] sigma: 2.6
[GPBoost] [Debug] lambda: 3
[GPBoost] [Debug] Initial approximate negative marginal log-li

### Vecchia (Correlation)

In [ ]:
cov_pars = [] 
aux_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
pred_latent = [] 
pred_var_latent = [] 
Time_vec = []
cov_par = pd.DataFrame([[ 1, 0.037, 6.4939e-06, 0.5, 0.5, 0.5, 0.5]],
                  columns=["sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[0.5] + [0]*(df.shape[1]-8)])
aux_par = pd.DataFrame([[2.6,3]])
for i in range(366, 572):
    print(i)
    df_train = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] >= "2024-06-30")]
    
    mask = (
        (df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i-100]))
    )

    # Apply filter
    df_filtered = df[mask]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["prcp"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "prcp"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    const_mask = np.all(X_train == X_train[0, :], axis=0)
    const_mask[0] = False
    X_train = X_train[:, ~const_mask]
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 5])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["prcp"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "prcp"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    X_test = X_test[:, ~const_mask]
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "iterative", gp_approx="vecchia_correlation_based")
        model.fit(X = X_train,y = y_train,
                 params={"maxit": 1000,"trace": True, "estimate_cov_par_index": [1,1,1,1,0,1,1], #"estimate_aux_pars": False, 
                         "init_aux_pars": aux_par.to_numpy().ravel().tolist(),
                         "std_dev": False, #"num_rand_vec_trace": 1000,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        print(aux_par)
        print(cov_par)
        cov_pars.append(cov_par)
        aux_pars.append(aux_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_neighbors = 30,vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "iterative", gp_approx="vecchia_correlation_based")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,0,1,1], 
                          "init_aux_pars": aux_par.to_numpy().ravel().tolist(),"std_dev": False,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])
    pred_Linear_latent_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_response = False,predict_var=True)
    pred_latent.append(pred_Linear_latent_Model["mu"])
    pred_var_latent.append(pred_Linear_latent_Model["var"])

366
Re-train model
[GPBoost] [Info] Use GPU
[GPBoost] [Warning] The linear regression covariate data matrix (fixed effect) is rank deficient. This is not necessarily a problem when using gradient descent. If this is not desired, consider dropping some columns / covariates 
[GPBoost] [Debug] GPModel: initial parameters: 
[GPBoost] [Debug] cov_pars[0]: 21.3
[GPBoost] [Debug] cov_pars[1]: 0.037
[GPBoost] [Debug] cov_pars[2]: 6.4939e-06
[GPBoost] [Debug] cov_pars[3]: 0.5
[GPBoost] [Debug] cov_pars[4]: 1.5
[GPBoost] [Debug] cov_pars[5]: 0.5
[GPBoost] [Debug] cov_pars[6]: 0.5
[GPBoost] [Debug] beta[0]: 0.5
[GPBoost] [Debug] beta[1]: 0
[GPBoost] [Debug] beta[2]: 0
[GPBoost] [Debug] beta[3]: 0
[GPBoost] [Debug] beta[4]: 0
[GPBoost] [Debug] Note: only the first 5 linear regression coefficients are shown 
[GPBoost] [Debug] sigma: 2.6
[GPBoost] [Debug] lambda: 2
[GPBoost] [Info] Launch 176314 128 46080
[GPBoost] [Debug] Initial approximate negative marginal log-likelihood: 207674
[GPBoost] [Debug

### FITC (kMeans++)

In [14]:
cov_pars = [] 
aux_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
pred_latent = [] 
pred_var_latent = [] 
Time_vec = []
cov_par = pd.DataFrame([[ 1, 0.037, 6.4939e-06, 0.5, 0.5, 0.5, 0.5]],
                  columns=["sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[0.5] + [0]*(df.shape[1]-8)])
aux_par = pd.DataFrame([[2.6,3]])
for i in range(366, 572):
    print(i)
    df_train = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] >= "2024-06-30")]
    
    mask = (
        (df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i-100]))
    )

    # Apply filter
    df_filtered = df[mask]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["prcp"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "prcp"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    const_mask = np.all(X_train == X_train[0, :], axis=0)
    const_mask[0] = False
    X_train = X_train[:, ~const_mask]
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 5])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["prcp"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "prcp"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    X_test = X_test[:, ~const_mask]
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_ind_points = 500,ind_points_selection = "kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                 params={"maxit": 1000,"trace": True, "estimate_cov_par_index": [1,1,1,1,0,1,1], #"estimate_aux_pars": False, 
                         "init_aux_pars": aux_par.to_numpy().ravel().tolist(),
                         "std_dev": False, #"num_rand_vec_trace": 1000,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        print(aux_par)
        print(cov_par)
        cov_pars.append(cov_par)
        aux_pars.append(aux_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_ind_points = 500,ind_points_selection = "kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,0,1,1], 
                          "init_aux_pars": aux_par.to_numpy().ravel().tolist(),"std_dev": False,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])
    pred_Linear_latent_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_response = False,predict_var=True)
    pred_latent.append(pred_Linear_latent_Model["mu"])
    pred_var_latent.append(pred_Linear_latent_Model["var"])

366
2024-09-23 00:00:00
Re-train model
[GPBoost] [Info] Use GPU
[GPBoost] [Debug] Starting kmeans++ algorithm for determining inducing points 
[GPBoost] [Debug] Inducing points have been determined 
[GPBoost] [Warning] The linear regression covariate data matrix (fixed effect) is rank deficient. This is not necessarily a problem when using gradient descent. If this is not desired, consider dropping some columns / covariates 
[GPBoost] [Debug] GPModel: initial parameters: 
[GPBoost] [Debug] cov_pars[0]: 2
[GPBoost] [Debug] cov_pars[1]: 0.037
[GPBoost] [Debug] cov_pars[2]: 6.4939e-06
[GPBoost] [Debug] cov_pars[3]: 0.5
[GPBoost] [Debug] cov_pars[4]: 0.5
[GPBoost] [Debug] cov_pars[5]: 0.5
[GPBoost] [Debug] cov_pars[6]: 1
[GPBoost] [Debug] beta[0]: 0.5
[GPBoost] [Debug] beta[1]: 0
[GPBoost] [Debug] beta[2]: 0
[GPBoost] [Debug] beta[3]: 0
[GPBoost] [Debug] beta[4]: 0
[GPBoost] [Debug] Note: only the first 5 linear regression coefficients are shown 
[GPBoost] [Debug] sigma: 2.1
[GPBoost] [Deb

[GPBoost] [Fatal] NaN occured in gradient wrt covariance / auxiliary parameter number 1 (counting starts at 1, total nb. par. = 9) 


Error occurred: NaN occured in gradient wrt covariance / auxiliary parameter number 1 (counting starts at 1, total nb. par. = 9) 
3098.9917891025543
           sigma    lambda
Param.  3.294798  1.643419
           sigma2         a             c     alpha   nu      beta     delta
Param.  24.282849  0.086773  4.257603e-07  0.177402  0.5  0.775437  0.434942
[[ 2.65379581  4.20533332  3.61333311  8.57024537 12.94070768  9.06936759]
 [ 0.68292678  0.93193877  1.07452464  3.73982272  6.93407765  2.55757666]
 [ 1.73956297  4.32386331  3.61053898  9.15800373 12.71590372  9.17115274]]
[[0.70609701 1.02727045 1.16424835 4.31840905 7.74233292 2.9975346 ]
 [0.71884114 1.04719279 1.17505681 4.23977021 7.7944532  2.99964102]
 [0.66777574 0.96733717 1.13159686 4.55508589 7.58611627 2.99119717]]
[[0.25385319 0.53747279 0.55158742 3.69223762 7.38617299 2.48999716]
 [0.26799834 0.55645765 0.56423369 3.61163696 7.43495661 2.49144681]
 [0.21223937 0.48059663 0.51662014 3.94409297 7.24223777 2.48883402]]
[

[GPBoost] [Fatal] NaN occured in gradient wrt covariance / auxiliary parameter number 1 (counting starts at 1, total nb. par. = 9) 


Error occurred: NaN occured in gradient wrt covariance / auxiliary parameter number 1 (counting starts at 1, total nb. par. = 9) 
4516.493912220001
           sigma    lambda
Param.  4.304913  1.607999
           sigma2         a             c    alpha   nu      beta     delta
Param.  35.192914  0.063868  1.673335e-07  0.23284  0.5  0.687884  0.269165
[[112.63871014 101.63138605  99.61370227 101.87788371 100.74138036
  103.88478501]
 [102.07970272 101.85478959 101.05667284  98.43576017  96.86602211
  100.05085568]
 [109.79283044  95.16583105  95.2030352   96.47471352  91.77168646
   98.59755317]]
[[102.7968268  101.3272692  100.7088905   98.30801841  96.53491728
   99.94566014]
 [103.76946942 102.19937773 101.41786148  98.97842988  97.14858394
  100.71306756]
 [ 99.90456589  98.73884136  98.59209126  96.29101287  94.70856947
   97.65706052]]
[[79.0549814  77.81240504 77.31789363 75.02414585 73.37637072 76.52637495]
 [80.03332946 78.61191363 78.01952228 75.74776152 73.96109874 77.283698

### FITC (space-time-separated kMeans++)

In [ ]:
cov_pars = [] 
aux_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
pred_latent = [] 
pred_var_latent = [] 
Time_vec = []
cov_par = pd.DataFrame([[ 1, 0.037, 6.4939e-06, 0.5, 0.5, 0.5, 0.5]],
                  columns=["sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[0.5] + [0]*(df.shape[1]-8)])
aux_par = pd.DataFrame([[2.6,3]])
for i in range(366, 572):
    print(i)
    df_train = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] >= "2024-06-30")]
    
    mask = (
        (df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i-100]))
    )

    # Apply filter
    df_filtered = df[mask]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["prcp"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "prcp"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    const_mask = np.all(X_train == X_train[0, :], axis=0)
    const_mask[0] = False
    X_train = X_train[:, ~const_mask]
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 5])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["prcp"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "prcp"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    X_test = X_test[:, ~const_mask]
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_ind_points = 500,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                 params={"maxit": 1000,"trace": True, "estimate_cov_par_index": [1,1,1,1,0,1,1], #"estimate_aux_pars": False, 
                         "init_aux_pars": aux_par.to_numpy().ravel().tolist(),
                         "std_dev": False, #"num_rand_vec_trace": 1000,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        print(aux_par)
        print(cov_par)
        cov_pars.append(cov_par)
        aux_pars.append(aux_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_ind_points = 500,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,0,1,1], 
                          "init_aux_pars": aux_par.to_numpy().ravel().tolist(),"std_dev": False,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])
    pred_Linear_latent_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_response = False,predict_var=True)
    pred_latent.append(pred_Linear_latent_Model["mu"])
    pred_var_latent.append(pred_Linear_latent_Model["var"])

366
Re-train model
[GPBoost] [Info] Use GPU
[GPBoost] [Debug] Starting space-time separated kmeans++ algorithm for determining inducing points 
[GPBoost] [Info] Number of time ip 20 number of space ip 26 and total 520
[GPBoost] [Debug] Inducing points have been determined 
[GPBoost] [Warning] The linear regression covariate data matrix (fixed effect) is rank deficient. This is not necessarily a problem when using gradient descent. If this is not desired, consider dropping some columns / covariates 
[GPBoost] [Debug] GPModel: initial parameters: 
[GPBoost] [Debug] cov_pars[0]: 12
[GPBoost] [Debug] cov_pars[1]: 0.037
[GPBoost] [Debug] cov_pars[2]: 6.4939e-06
[GPBoost] [Debug] cov_pars[3]: 0.5
[GPBoost] [Debug] cov_pars[4]: 1.5
[GPBoost] [Debug] cov_pars[5]: 0.5
[GPBoost] [Debug] cov_pars[6]: 0.5
[GPBoost] [Debug] beta[0]: 0.5
[GPBoost] [Debug] beta[1]: 0
[GPBoost] [Debug] beta[2]: 0
[GPBoost] [Debug] beta[3]: 0
[GPBoost] [Debug] beta[4]: 0
[GPBoost] [Debug] Note: only the first 5 linear 

### VIF

In [6]:
cov_pars = [] 
aux_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
pred_latent = [] 
pred_var_latent = [] 
Time_vec = []
cov_par = pd.DataFrame([[ 1, 0.037, 6.4939e-06, 0.5, 0.5, 0.5, 0.5]],
                  columns=["sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[0.5] + [0]*(df.shape[1]-8)])
aux_par = pd.DataFrame([[2.6,3]])
for i in range(366, 572):
    print(i)
    df_train = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] >= "2024-06-30")]
    
    mask = (
        (df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i-100]))
    )

    # Apply filter
    df_filtered = df[mask]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["prcp"]].to_numpy().ravel()
    X_train = df_train_filtered.drop(columns=["t", "date", "prcp"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    const_mask = np.all(X_train == X_train[0, :], axis=0)
    const_mask[0] = False
    X_train = X_train[:, ~const_mask]
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 5])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["prcp"]].to_numpy().ravel()
    X_test = df_test.drop(columns=["t", "date", "prcp"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    X_test = X_test[:, ~const_mask]
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_ind_points = 500,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia_correlation_based",num_neighbors = 30,vecchia_ordering = "random")
        model.fit(X = X_train,y = y_train,
                 params={"maxit": 1000,"trace": True, "estimate_cov_par_index": [1,1,1,1,0,1,1], #"estimate_aux_pars": False, 
                         "init_aux_pars": aux_par.to_numpy().ravel().tolist(),
                         "std_dev": False, #"num_rand_vec_trace": 1000,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        print(aux_par)
        print(cov_par)
        cov_pars.append(cov_par)
        aux_pars.append(aux_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="zero_censored_power_transformed_normal",seed = 2,num_ind_points = 500,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia_correlation_based",num_neighbors = 30,vecchia_ordering = "random")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,0,1,1], 
                          "init_aux_pars": aux_par.to_numpy().ravel().tolist(),"std_dev": False,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])
    pred_Linear_latent_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_response = False,predict_var=True)
    pred_latent.append(pred_Linear_latent_Model["mu"])
    pred_var_latent.append(pred_Linear_latent_Model["var"])

366
2024-09-23 00:00:00
Re-train model
[GPBoost] [Info] Use GPU
[GPBoost] [Info] Number of time ip 5 number of space ip 94 and total 470
[GPBoost] [Warning] The linear regression covariate data matrix (fixed effect) is rank deficient. This is not necessarily a problem when using gradient descent. If this is not desired, consider dropping some columns / covariates 
[GPBoost] [Debug] GPModel: initial parameters: 
[GPBoost] [Debug] cov_pars[0]: 1
[GPBoost] [Debug] cov_pars[1]: 0.037
[GPBoost] [Debug] cov_pars[2]: 6.4939e-06
[GPBoost] [Debug] cov_pars[3]: 0.2
[GPBoost] [Debug] cov_pars[4]: 0.5
[GPBoost] [Debug] cov_pars[5]: 0.2
[GPBoost] [Debug] cov_pars[6]: 0.5
[GPBoost] [Debug] beta[0]: 0.5
[GPBoost] [Debug] beta[1]: 0
[GPBoost] [Debug] beta[2]: 0
[GPBoost] [Debug] beta[3]: 0
[GPBoost] [Debug] beta[4]: 0
[GPBoost] [Debug] Note: only the first 5 linear regression coefficients are shown 
[GPBoost] [Debug] sigma: 2.6
[GPBoost] [Debug] lambda: 3
[GPBoost] [Info] Launch 176258 128 46080
[GPBo

## Bernoulli Likelihood 

### Example for VIF

In [ ]:
cov_pars = [] 
aux_pars = [] 
coef_pars = [] 
pred_mu = [] 
pred_var = []
pred_latent = [] 
pred_var_latent = [] 
Time_vec = []
cov_par = pd.DataFrame([[ 1, 0.037, 6.4939e-06, 0.5, 0.5, 0.5, 0.5]],
                  columns=["sigma2", "a", "c", "alpha", "nu", "beta", "delta"])
coef = pd.DataFrame([[0.5] + [0]*(df.shape[1]-8)])
aux_par = pd.DataFrame([[2.6,3]])
for i in range(366, 572):
    print(i)
    df_train = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] >= "2024-06-30")]
    
    mask = (
        (df["date"] <= np.max(data_complete["date"][data_complete["t"] == i])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i-100]))
    )

    # Apply filter
    df_filtered = df[mask]
    # Use anti-join logic: remove rows whose (X, Y) are in stations_to_remove
    mask = df_train.set_index(["X", "Y"]).index.isin(stations_to_remove.set_index(["X", "Y"]).index)

    df_train_filtered = df_train[~mask].copy()
    coords_train = df_train_filtered[["t", "X", "Y"]].to_numpy()
    y_train = df_train_filtered[["prcp"]].to_numpy().ravel()
    y_train = np.where(y_train.to_numpy() > 0, 1, 0)
    X_train = df_train_filtered.drop(columns=["t", "date", "prcp"]).copy()
    X_train.insert(0, "ones", 1)
    X_train = X_train.to_numpy()
    const_mask = np.all(X_train == X_train[0, :], axis=0)
    const_mask[0] = False
    X_train = X_train[:, ~const_mask]
    df_test = df[(df["date"] <= np.max(data_complete["date"][data_complete["t"] == i + 5])) & (df["date"] > np.max(data_complete["date"][data_complete["t"] == i]))]
    coords_test = df_test[["t", "X", "Y"]].to_numpy()
    y_test = df_test[["prcp"]].to_numpy().ravel()
    y_test = np.where(y_test.to_numpy() > 0, 1, 0)
    X_test = df_test.drop(columns=["t", "date", "prcp"]).copy()
    X_test.insert(0, "ones", 1)
    X_test = X_test.to_numpy()
    X_test = X_test[:, ~const_mask]
    if i in doys_shifted:
        print("Re-train model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="bernoulli_logit",seed = 2,num_ind_points = 500,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia_correlation_based",num_neighbors = 30,vecchia_ordering = "random")
        model.fit(X = X_train,y = y_train,
                 params={"maxit": 1000,"trace": True, "estimate_cov_par_index": [1,1,1,1,0,1,1], #"estimate_aux_pars": False, 
                         "init_aux_pars": aux_par.to_numpy().ravel().tolist(),
                         "std_dev": False, #"num_rand_vec_trace": 1000,
                         "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        end_time = time.time()
        Time_vec.append(end_time-start_time)
        print(end_time-start_time)
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        print(aux_par)
        print(cov_par)
        cov_pars.append(cov_par)
        aux_pars.append(aux_par)
        coef = model.get_coef()
        coef_pars.append(coef)
    else:
        print("Re-define model")
        start_time = time.time()
        model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="bernoulli_logit",seed = 2,num_ind_points = 500,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "iterative", gp_approx="full_scale_vecchia_correlation_based",num_neighbors = 30,vecchia_ordering = "random")
        model.fit(X = X_train,y = y_train,
                  params={"maxit": 0,"trace": False, "estimate_cov_par_index": [1,1,1,1,0,1,1], 
                          "init_aux_pars": aux_par.to_numpy().ravel().tolist(),"std_dev": False,
                          "init_cov_pars": cov_par.to_numpy().ravel().tolist(), "init_coef": coef.to_numpy().ravel().tolist()})
        cov_par = model.get_cov_pars()
        aux_par = model.get_aux_pars()
        coef = model.get_coef()
        end_time = time.time()
        
    pred_Linear_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_var=True) 
    pred_mu.append(pred_Linear_Model["mu"])
    pred_var.append(pred_Linear_Model["var"])
    pred_Linear_latent_Model = model.predict(X_pred=X_test,gp_coords_pred=coords_test,y=y_train,predict_response = False,predict_var=True)
    pred_latent.append(pred_Linear_latent_Model["mu"])
    pred_var_latent.append(pred_Linear_latent_Model["var"])